In [23]:
import pandas  as pd
import re
import os
import matplotlib.pyplot as plt

In [24]:
patron = r"result(.*?)_"
root='bases'

In [25]:
bases=os.listdir(root)
models=set([re.match(patron,x).group(1) for x in bases])
print(models)
dataset={}
for model in models:
    dataset[model]=[pd.read_csv(root+"/"+x,index_col=0) for x in bases if model in x]


{'Llama-3.2-1B-Instruct', 'Qwen2.5-1.5B-Instruct', 'DeepSeek-R1-Distill-Qwen-1.5B'}


In [26]:
prompts=dataset['Qwen2.5-1.5B-Instruct'][0]['prompt'].copy()
for k in dataset.keys():
    r=prompts.copy()
    dbs=dataset[k]
    for db  in dbs:
        r=pd.merge(r,db,on='prompt')
        print(r.shape)
    dataset[k]=r

(1000, 5)
(1000, 9)
(1000, 13)
(1000, 17)
(1000, 5)
(1000, 9)
(1000, 13)
(1000, 17)
(1000, 5)
(1000, 9)
(1000, 13)
(1000, 17)


In [ ]:
#revisamos  si  existen  respuestas  iguales
from itertools import product
llama=dataset['Qwen2.5-1.5B-Instruct']
print(llama.shape)
a={1,2,3,4}
b={1,2,3,4}
c=list(product(a,b))
c=[x for x in c if x[0]!=x[1]]
for par in c:
    print(llama[llama[f'response_{par[0]}']==llama[f'response_{par[1]}']].shape)

(1000, 17)
(195, 17)
(168, 17)
(180, 17)
(195, 17)
(184, 17)
(199, 17)
(168, 17)
(184, 17)
(179, 17)
(180, 17)
(199, 17)
(179, 17)


In [41]:
dataset['DeepSeek-R1-Distill-Qwen-1.5B'].columns

Index(['prompt', 'response_1', 'tiempo_1', 'ram_mb_1', 'gpu_mb_1',
       'response_2', 'tiempo_2', 'ram_mb_2', 'gpu_mb_2', 'response_3',
       'tiempo_3', 'ram_mb_3', 'gpu_mb_3', 'response_4', 'tiempo_4',
       'ram_mb_4', 'gpu_mb_4'],
      dtype='object')

In [42]:
#funcion para  crear  las  4   respuestas
def make4(db:pd.DataFrame):
    a= db.columns
    a=[x for x in a if 'response' in x]
    a.insert(0,'prompt')
    db1=db[a].copy()
    return db1

In [43]:
tocalify={}
root=r'C:\Users\Gabo\Desktop\PAPPER Rlaif\codPreferences\calify/'
for k in dataset.keys():
    r=make4(dataset[k])
    r.to_csv(root+k+".csv")